# Krea 2 LoRA 学習ノートブック (fal互換・完全自動) v4.3

**v4.3の構造**: あなたが操作するのは**最初の3セルだけ** (トークン入力・Drive許可・設定確認)。以降は完全自動で、複数データセットを連続学習し、各LoRAをDriveに保存します。

**事前準備** (一度だけ):
1. **HuggingFaceトークン** — `krea/Krea-2-Raw` / `Qwen/Qwen3-VL-4B-Instruct` / `Qwen/Qwen-Image` のライセンス承認が必要
2. **データセットZIP** — 画像15〜30枚+同名.txtキャプション を `MyDrive/krea2/` に配置
3. **設定ファイル** `MyDrive/krea2/jobs.json` (例):
```json
{
  "resolution": 1024,
  "lr": 1e-4,
  "rank": 32,
  "jobs": [
    {"trigger": "satou_kibi_style", "zip": "/content/drive/MyDrive/krea2/satoukibi_20260808_148.zip"}
  ]
}
```
- **steps は省略可**: 画像枚数から自動決定します (≤10:400 / ≤30:600 / ≤50:1000 / ≤100:1250 / ≤150:1500 / >150:1800)。ジョブごとに `"steps": N` を書けば上書き、グローバルに `"steps"` を書けば全ジョブ固定になります
- trigger はキャプション先頭のタグと**完全一致**させること (例: `satou_kibi_style`)

**実行手順**: 下の3セルを実行(操作) → 残りのセルを上から順にすべて実行。あとは放置でOK (A100: 1ジョブ約1時間)。
**注意**: 学習中はブラウザのタブを閉じないこと。途中で切れても再実行すればチェックポイントから続きを再開します。


## 1. あなたの操作 (ここだけ・最初に全部済ませる)

### 1-1. HuggingFaceトークン

Colabの左サイドバーにある **🔑（シークレット）** アイコンから  を登録しておくか、未登録の場合は実行時に入力プロンプトが表示されます。


In [ ]:
import getpass, os

token = None
# 1. Colabのシークレット機能 (🔑) から取得を試みる
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

# 2. 環境変数に既にあるか確認
if not token:
    token = os.environ.get("HF_TOKEN")

# 3. なければ入力プロンプトを表示 (非表示入力)
if not token:
    token = getpass.getpass("HuggingFace Access Token を入力してください (hf_...): ").strip()

if token:
    os.environ["HF_TOKEN"] = token
    try:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
    except Exception:
        pass
    print("トークン: OK (設定済み)")
else:
    print("⚠️ トークン未設定!")


トークン: OK (設定済み)


### 1-2. Google Driveマウント

ポップアップでGoogleアカウント選択・許可が出たら「許可」をクリック。画面の陰に隠れやすいので注意。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DriveマウントOK")

Mounted at /content/drive
DriveマウントOK


### 1-3. 設定読み込みと確認

`MyDrive/krea2/jobs.json` を読み込み、学習予定のジョブ一覧を表示します。内容を変えたい場合は jobs.json を編集して**このセルだけ再実行**してください。

In [ ]:
import json, os
CFG_PATH = "/content/drive/MyDrive/krea2/jobs.json"
assert os.path.exists(CFG_PATH), f"設定ファイルが見つかりません: {CFG_PATH} (MyDrive/krea2/jobs.json を作成してください)"
cfg = json.load(open(CFG_PATH, encoding="utf-8"))
JOBS = cfg["jobs"]
RESOLUTION = cfg.get("resolution", 1024)
STEPS      = cfg.get("steps")  # 未指定なら calc_steps() が画像枚数から自動決定
LR         = cfg.get("lr", 1e-4)
RANK       = cfg.get("rank", 32)
assert JOBS, "JOBSが空です"
for j in JOBS:
    assert j.get("trigger") and j.get("zip"), "JOBSの各要素に trigger と zip が必要です"
steps_disp = STEPS if STEPS else "AUTO(画像枚数ベース)"
print("ジョブ数:", len(JOBS), "/ 解像度:", RESOLUTION, "px / ステップ:", steps_disp, "/ lr:", LR, "/ rank:", RANK)
for j in JOBS:
    print(" -", j["trigger"], "<-", j["zip"].rsplit("/", 1)[-1])
print()
print("問題なければ次のセクションへ。変更したい場合は jobs.json を編集して再実行。")

ジョブ数: 6 / 解像度: 1024 px / ステップ: AUTO(画像枚数ベース) / lr: 0.0005 / rank: 32
 - do_style <- do_20260812_30.zip
 - pote_style <- pote_20260812_30.zip
 - yatsuki_style <- yatsuki_20260812_30.zip
 - nama_style <- nama_20260812_30.zip
 - oninamako_style <- oninamako_20260812_30.zip
 - karei_style <- karei_20260813_30.zip

問題なければ次のセクションへ。変更したい場合は jobs.json を編集して再実行。


## 2. 環境構築 (自動・一度実行すればスキップ)

numpy 2.x環境と両立させるため、requirementsのscipy固定(numpy<2専用)を置換してからインストールします。

In [ ]:
import os, sys
print("Python:", sys.version.split()[0])
if not os.path.exists("/content/.env_done"):
    if not os.path.exists("/content/ai-toolkit"):
        os.system("git clone -q https://github.com/ostris/ai-toolkit.git /content/ai-toolkit")
        print("ai-toolkit cloned")
    else:
        print("ai-toolkit already exists")
    # requirements修正: numpy 2.x環境と両立させる (scipy 1.12はnumpy<2専用のため置換)
    p_req2 = "/content/ai-toolkit/requirements.txt"
    s_req2 = open(p_req2, encoding="utf-8").read()
    if "scipy==1.12.0" in s_req2:
        open(p_req2, "w", encoding="utf-8").write(s_req2.replace("scipy==1.12.0", "scipy>=1.14"))
        print("requirements修正OK (scipy>=1.14)")
    else:
        print("requirements: 修正済み")
    # torch (CUDA 12.6)
    os.system("pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    # 依存一式
    os.system("pip install -q -r /content/ai-toolkit/requirements.txt")
    # torch/torchvision/torchaudio をCUDA 12.6版で統一 (requirementsがPyPI版(cu128)を混入させるため)
    os.system("pip install -q --force-reinstall --no-deps torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    # ディスク節約: pipキャッシュ削除 (モデルDL時の容量警告を防ぐ)
    os.system("pip cache purge")
    open("/content/.env_done", "w").write("done")
    print("環境構築完了")
else:
    print("環境構築: 済み (スキップ)")
import torch
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1fGB" % (props.total_memory/1e9))

Python: 3.12.13
ai-toolkit cloned
requirements修正OK (scipy>=1.14)
環境構築完了
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4GB


## 3. 必須パッチ適用 (自動・一度適用すればスキップ)

1. **quanto dequantizeフォールバック**: FP8非対応GPU (T4/V100/A100/3080Ti等) でqfloat8がネイティブクラッシュする問題の回避
2. **fal互換264モジュール**: AI-Toolkitのフィルタで漏れるグローバル8層 (img_in/final_layer/time_embed等) を学習対象に追加

In [ ]:
import os
if not os.path.exists("/content/.patches_done"):
    # --- パッチ1: quanto dequantize フォールバック ---
    import optimum.quanto
    p1 = os.path.join(os.path.dirname(optimum.quanto.__file__), "tensor", "qtensor_func.py")
    src = open(p1, encoding="utf-8").read()
    old = '''        elif isinstance(other, QBytesTensor):
            if isinstance(input, QBytesTensor):
                output = torch.ops.quanto.qbytes_mm(input._data, other._data, input._scale * other._scale)
            else:
                output = torch.ops.quanto.qbytes_mm(input, other._data, other._scale)'''
    new = '''        elif isinstance(other, QBytesTensor):
            # ROCm/Ampere非対応qbytes_mm回避: dequantizeして標準matmul
            output = torch.matmul(input, other.dequantize().t())'''
    assert old in src, "patch1 target not found (quanto version違い?)"
    open(p1, "w", encoding="utf-8").write(src.replace(old, new))
    print("patch1 (quanto dequantize fallback): OK")

    # --- パッチ2: fal互換264モジュール ---
    p2 = "/content/ai-toolkit/toolkit/lora_special.py"
    src = open(p2, encoding="utf-8").read()
    a = "if transformer_block_names is not None:"
    b = "if transformer_block_names is not None and type(root_module).__name__ != 'SingleStreamDiT':"
    assert a in src, "patch2a target not found"
    src = src.replace(a, b)
    c = "if hasattr(root_module, 'blocks'):"
    d = "if hasattr(root_module, 'blocks') and type(root_module).__name__ != 'SingleStreamDiT':"
    assert c in src, "patch2b target not found"
    src = src.replace(c, d)
    open(p2, "w", encoding="utf-8").write(src)
    print("patch2 (264 modules): OK")

    # --- パッチ3: lr_scheduler の T_0 を尊重 (cosine_with_restarts で T_0/T_mult 指定可能に) ---
    # AI-Toolkit は total_iters(=steps) を自動注入して T_0 を上書きするため、
    # lr_scheduler_params で T_0 を明示指定した場合は total_iters を破棄する
    p3 = "/content/ai-toolkit/toolkit/scheduler.py"
    src = open(p3, encoding="utf-8").read()
    e = '''    elif name == "cosine_with_restarts":
        if 'total_iters' in kwargs:
            kwargs['T_0'] = kwargs.pop('total_iters')'''
    f = '''    elif name == "cosine_with_restarts":
        if 'total_iters' in kwargs and 'T_0' not in kwargs:
            kwargs['T_0'] = kwargs.pop('total_iters')
        elif 'total_iters' in kwargs:
            kwargs.pop('total_iters')'''
    assert e in src, "patch3 target not found (scheduler version違い?)"
    src = src.replace(e, f)
    open(p3, "w", encoding="utf-8").write(src)
    print("patch3 (scheduler T_0 respect): OK")

    open("/content/.patches_done", "w").write("done")
    print("パッチ適用完了")
else:
    print("パッチ: 済み (スキップ)")

patch1 (quanto dequantize fallback): OK
patch2 (264 modules): OK
patch3 (scheduler T_0 respect): OK
パッチ適用完了


## 4. モデルダウンロード (自動・DL済みならスキップ)

RAW 26GB + テキストエンコーダ + VAE。Colabの高速回線なら約2分です (hf_hub の高速ダウンロードを使用)。

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("/content/models/krea2_raw", exist_ok=True)
RAW = "/content/models/krea2_raw/raw.safetensors"
if not os.path.exists(RAW):
    p = hf_hub_download("krea/Krea-2-Raw", "raw.safetensors")
    shutil.copy(p, RAW)
    print("RAW: ダウンロード完了")
print("RAW: %.1f GB" % (os.path.getsize(RAW)/1e9))
if not os.path.exists("/content/models/qwen3vl/config.json"):
    snapshot_download("Qwen/Qwen3-VL-4B-Instruct", local_dir="/content/models/qwen3vl")
if not os.path.exists("/content/models/qwen_image/vae"):
    snapshot_download("Qwen/Qwen-Image", allow_patterns=["vae/*"], local_dir="/content/models/qwen_image")
# ディスク節約: HFキャッシュ削除 (/content/modelsに実体があるため不要。26GB分の二重化を解消)
if os.path.isdir("/root/.cache/huggingface"):
    os.system("rm -rf /root/.cache/huggingface")
    print("HFキャッシュ削除 (ディスク節約)")
print("MODELS_DONE")

raw.safetensors: reconstructing file:   0%|          |  0.00B / 26.3GB            

raw.safetensors: downloading bytes:           |  0.00B            

RAW: ダウンロード完了
RAW: 26.3 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

HFキャッシュ削除 (ディスク節約)
MODELS_DONE


## 5. 関数定義 (自動・定義するだけ)

ここでは処理を関数として定義します。実際の実行はセクション6のループが行います。

In [ ]:
import glob, os, shutil, zipfile

META_TAGS = {"character name", "character", "name", "symbols", "text", "english text",
             "dated", "score", "no humans", "commentary", "translated", "artist name"}

def calc_steps(n_imgs):
    # 画像枚数から学習ステップ数を自動決定 (Krea公式+コミュニティ実績・2026-08-08確定・LR 1e-4 前提)
    if n_imgs <= 10:
        return 400
    if n_imgs <= 30:
        return 1000
    if n_imgs <= 50:
        return 1000
    if n_imgs <= 100:
        return 1250
    if n_imgs <= 150:
        return 1500
    return 1800

def split_tags_and_prose(caption):
    parts = [p.strip() for p in caption.split(",")]
    prose_idx = None
    for i, p in enumerate(parts):
        words = p.split()
        if len(words) >= 4 and p[0].isupper() and not p.isupper() and not p[0].isdigit():
            prose_idx = i
            break
    if prose_idx is None:
        return parts, []
    return parts[:prose_idx], parts[prose_idx:]

def fix_caption(caption, trigger):
    tags, prose = split_tags_and_prose(caption)
    if trigger and trigger in tags:
        tags = [t for t in tags if t != trigger]
    tags = [t for t in tags if t.lower() not in META_TAGS]
    keep = []
    for t in tags:
        tlow = t.lower()
        if len(t.split()) == 1 and any(tlow in o.lower().split() and o.lower() != tlow for o in tags):
            continue
        keep.append(t)
    seen, deduped = set(), []
    for t in keep:
        if t.lower() not in seen:
            seen.add(t.lower())
            deduped.append(t)
    return ", ".join([trigger] + prose + deduped)

def prepare_dataset(zip_path, trigger):
    assert os.path.exists(zip_path), f"ZIPが見つかりません: {zip_path}"
    os.system("rm -rf /content/dataset")
    os.makedirs("/content/dataset", exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content/dataset")
    for f in glob.glob("/content/dataset/**/*", recursive=True):
        if os.path.isfile(f) and os.path.dirname(f) != "/content/dataset":
            shutil.move(f, os.path.join("/content/dataset", os.path.basename(f)))
    imgs = sorted(glob.glob("/content/dataset/*.png") + glob.glob("/content/dataset/*.jpg")
                  + glob.glob("/content/dataset/*.jpeg") + glob.glob("/content/dataset/*.webp"))
    caps = sorted(glob.glob("/content/dataset/*.txt"))
    assert len(imgs) >= 10, "画像が少なすぎます (10枚以上推奨)"
    changed = 0
    for txt in caps:
        raw = open(txt, encoding="utf-8", errors="ignore").read().strip()
        fixed = fix_caption(raw, trigger)
        if fixed != raw:
            changed += 1
        open(txt, "w", encoding="utf-8").write(fixed + "\n")
    print(f"データセット: 画像{len(imgs)}枚 / キャプション{len(caps)}個 (修正{changed}個) / trigger={trigger}")
    if caps:
        print("例:", open(caps[0], encoding="utf-8").read().strip()[:150], "...")
    return len(imgs)

In [ ]:
import glob, json, os, re, shutil, struct, torch

# ---------- config生成 ----------
def make_config(trigger, res, steps, lr, rank):
    yaml = f'''---
job: extension
config:
  name: "{trigger}_colab"
  process:
    - type: 'sd_trainer'
      training_folder: "output/{trigger}_colab"
      device: cuda:0
      network:
        type: "lora"
        linear: {rank}
        linear_alpha: {rank}
      save:
        dtype: float16
        # 100刻み保存(2026-08-10 復活): 崩壊ラインはデータセット依存のため、
        # 途中チェックポイントが保険になる(satou_kibi 1500steps 破損の教訓)
        save_every: 100
        max_step_saves_to_keep: 20
      datasets:
        - folder_path: "/content/dataset"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [{res}]
      train:
        batch_size: 1
        cache_text_embeddings: true
        steps: {steps}
        gradient_accumulation: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw"
        lr: {lr}
        dtype: bf16
        timestep_type: linear
        lr_scheduler: "cosine_with_restarts"
        lr_scheduler_params:
          T_0: 100
          T_mult: 2
      model:
        name_or_path: "/content/models/krea2_raw"
        arch: "krea2"
        quantize: {quant}
        qtype: "qfloat8"
        layer_offloading: {offload}
        layer_offloading_transformer_percent: 0.8
        low_vram: {lowvram}
        vae_device: cpu
        model_kwargs:
          checkpoint_filename: "raw.safetensors"
          text_encoder_path: "/content/models/qwen3vl"
          vae_path: "/content/models/qwen_image"
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: {res}
        height: {res}
        prompts:
          - "{trigger}, 1girl, solo, A sample illustration in the style of {trigger}"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 3
        sample_steps: 25
meta:
  name: "[name]"
'''
    open("/content/ai-toolkit/krea2_colab.yaml", "w", encoding="utf-8").write(yaml)
    print(f"config書込OK: resolution [{res}] / quantize {quant} / {steps}steps / lr {lr}")

# ---------- 学習 ----------
def train(trigger):
    os.chdir("/content/ai-toolkit")
    os.system("python -u run.py krea2_colab.yaml")
    outs = sorted(glob.glob(f"output/{trigger}_colab/{trigger}_colab/*.safetensors"))
    print(f"学習出力: {len(outs)}個のファイル")
    for o in outs[-3:]:
        print(" ", os.path.basename(o))
    assert outs, "学習出力がありません。上のログにエラーが出ていないか確認してください"

# ---------- キー変換 (AI-Toolkit形式 → ComfyUI形式) ----------
PREFIXES = ["base_model.model.", "diffusion_model."]
BLOCK_SUB = {"attn.wq": "attn.to_q", "attn.wk": "attn.to_k", "attn.wv": "attn.to_v",
             "attn.gate": "attn.to_gate", "attn.wo": "attn.to_out.0",
             "mlp.gate": "ff.gate", "mlp.up": "ff.up", "mlp.down": "ff.down"}
BASIC_MAP = {"first": "img_in", "tmlp.0": "time_embed.linear_1", "tmlp.2": "time_embed.linear_2",
             "tproj.1": "time_mod_proj", "txtmlp.1": "txt_in.linear_1", "txtmlp.3": "txt_in.linear_2",
             "txtfusion.projector": "text_fusion.projector", "last.linear": "final_layer.linear"}

def convert_key(k):
    prefix = None
    for p in PREFIXES:
        if k.startswith(p):
            prefix = p
            break
    if prefix is None:
        return None
    rest = k[len(prefix):]
    dot = rest.rfind(".lora_")
    if dot < 0:
        return None
    suffix = rest[dot:]
    body = rest[:dot]
    parts = body.split(".")
    if parts[0] == "blocks" and len(parts) >= 3:
        sub = ".".join(parts[2:])
        if sub not in BLOCK_SUB:
            return None
        return "transformer.transformer_blocks.{}.{}".format(parts[1], BLOCK_SUB[sub]) + suffix
    if parts[0] == "txtfusion":
        if parts[1] == "projector":
            return "transformer.text_fusion.projector" + suffix
        sub = ".".join(parts[3:])
        if sub not in BLOCK_SUB:
            return None
        return "transformer.text_fusion.{}.{}.{}".format(parts[1], parts[2], BLOCK_SUB[sub]) + suffix
    if body in BASIC_MAP:
        return "transformer." + BASIC_MAP[body] + suffix
    return None

def convert_file(src, dst):
    with open(src, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(n))
        tensors = f.read()
    new_header = {}
    converted = 0
    for k, meta in header.items():
        if k == "__metadata__":
            new_header[k] = meta
            continue
        nk = convert_key(k)
        if nk is None:
            print("skip:", k)
            continue
        new_header[nk] = meta
        converted += 1
    hb = json.dumps(new_header).encode()
    with open(dst, "wb") as f:
        f.write(struct.pack("<Q", len(hb)))
        f.write(hb)
        f.write(tensors)
    print(f"変換完了: {converted}キー -> {dst}")

def convert(trigger):
    # 全チェックポイント(step付き9桁ゼロ埋め)+最終ファイルを変換
    folder = f"/content/ai-toolkit/output/{trigger}_colab/{trigger}_colab"
    ckpts = sorted(glob.glob(os.path.join(folder, f"{trigger}_colab_*.safetensors")))
    final = os.path.join(folder, f"{trigger}_colab.safetensors")
    if os.path.exists(final):
        ckpts.append(final)
    assert ckpts, "学習出力が見つかりません (学習が失敗している可能性)"
    converted = []
    for ckpt in ckpts:
        base = os.path.basename(ckpt)
        m = re.search(r"_(\d{9})\.safetensors$", base)
        if m:
            step = int(m.group(1))
            out = f"/content/{trigger}_lora_converted_{step}.safetensors"
        else:
            step = None
            out = f"/content/{trigger}_lora_converted_final.safetensors"
        convert_file(ckpt, out)
        converted.append((step, out))
    return converted

# ---------- 納品 (Drive保存のみ。ダウンロードはDriveから) ----------
def deliver(trigger, steps, lr, converted, sweep_min=None):
    out_dir = "/content/drive/MyDrive/krea2/outputs"
    os.makedirs(out_dir, exist_ok=True)
    import datetime
    jst = datetime.timezone(datetime.timedelta(hours=9))
    dstr = datetime.datetime.now(jst).strftime("%Y%m%d")
    delivered = []
    for step, src in sorted(converted, key=lambda x: (x[0] is None, x[0] or 0)):
        if step is not None and sweep_min is not None and step < sweep_min:
            continue
        label = step if step is not None else steps
        dst = os.path.join(out_dir, f"{trigger}_{dstr}_{label}steps_lr{lr}.safetensors")
        shutil.copy(src, dst)
        delivered.append(os.path.basename(dst))
        print("Drive保存:", os.path.basename(dst), "(%.1f MB)" % (os.path.getsize(src)/1e6))
    print("納品数:", len(delivered))
    return delivered

print("関数定義OK (prepare_dataset / make_config / train / convert / deliver)")

関数定義OK (prepare_dataset / make_config / train / convert / deliver)


## 6. 全ジョブ実行 (自動・本丸)

JOBSの全データセットを順番に: **展開 → タグ修正 → 学習 → 変換 → Drive保存** します。
- 進捗は `JOB 1/4: inunoko` → `JOB_DONE: inunoko` の流れ
- 1ジョブがエラーしても**残りは続行**し、最後に `FAILED: [...]` を表示
- **再実行すると、完了済みジョブはスキップされ、失敗ジョブだけ最初からクリーンに学習し直します**
- 学習中の loss はリアルタイム表示 (A100: 1ジョブ約1時間)
- **ブラウザのタブを閉じないこと** (切断すると中断されます)

In [ ]:
import os, traceback

# GPU情報 (config用)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu} / VRAM: {vram_gb:.1f}GB")
res = RESOLUTION
if vram_gb < 22:
    res = 768
    print("VRAM 22GB未満: 768pxに固定")
if vram_gb >= 36:
    quant, offload, lowvram = "false", "false", "false"
    print("→ 構成: bf16 / オフロードなし")
else:
    quant, offload, lowvram = "true", "true", "true"
    print("→ 構成: qfloat8 / レイヤーオフロード0.8 / VAE cpu")

# 完了マーカー: 完了済みジョブは再実行時にスキップ (エラー分だけクリーン再学習)
DONE = "/content/jobs_done"
os.makedirs(DONE, exist_ok=True)

N = len(JOBS)
failed = []
for i, job in enumerate(JOBS, 1):
    trig, zip_ = job["trigger"], job["zip"]
    if os.path.exists(os.path.join(DONE, f"{trig}.done")):
        print(f"\nSKIP: {trig} は完了済み (再実行時はスキップ)")
        continue
    print(f"\n{'='*64}\nJOB {i}/{N}: {trig} ({zip_.rsplit('/', 1)[-1]})\n{'='*64}")
    # 未完了なら出力フォルダをクリーンにして「最初から」学習する
    os.system(f"rm -rf /content/ai-toolkit/output/{trig}_colab")
    try:
        n_imgs = prepare_dataset(zip_, trig)
        steps = job.get("steps") or STEPS or calc_steps(n_imgs)
        print(f"→ ステップ数: {steps} (画像{n_imgs}枚)")
        make_config(trig, res, steps, LR, RANK)
        train(trig)
        converted = convert(trig)
        deliver(trig, steps, LR, converted, sweep_min=job.get("sweep_min"))
        open(os.path.join(DONE, f"{trig}.done"), "w").write("done")
        print(f"JOB_DONE: {trig} ({i}/{N})")
    except Exception as e:
        # エラー時は記録のみ。再実行すると未完了ジョブだけ最初からやり直す
        print(f"!! JOB_ERROR: {trig} -> {type(e).__name__}: {e}")
        traceback.print_exc()
        failed.append(trig)

print("\n" + "="*64)
if failed:
    print(f"FAILED: {len(failed)}ジョブ -> {failed}")
    print("→ このセルを再実行すると、失敗ジョブだけクリーンにやり直します")
else:
    print("ALL_JOBS_DONE 🎉 全ジョブ完了!")
print("="*64)

GPU: NVIDIA A100-SXM4-40GB / VRAM: 42.4GB
→ 構成: bf16 / オフロードなし

JOB 1/6: do_style (do_20260812_30.zip)
データセット: 画像30枚 / キャプション30個 (修正30個) / trigger=do_style
例: do_style, A girl with short light blue hair and a large dark blue bow sits against a white background, winking her left eye while keeping her right bl ...
→ ステップ数: 1500 (画像30枚)
config書込OK: resolution [1024] / quantize false / 1500steps / lr 0.0005
学習出力: 15個のファイル
  do_style_colab_000001200.safetensors
  do_style_colab_000001300.safetensors
  do_style_colab_000001400.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_100.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_200.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_300.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_400.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_500.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_600.safetensors
変換完了: 528キー -> /content/do_style_lora_converted_700.safetensors
変換完了: 528キー -

## 完了! 次のステップ

1. 各LoRAは `MyDrive/krea2/outputs/` に保存されています (ファイルブラウザまたはDriveから取得)
2. サーバーへ配置: `~/ComfyUI/models/loras/` にscp、またはファイルを渡してもらえればこちらで配置します
3. 生成時の推奨: **Krea-2-Turbo** に適用 / strength 1.0 / プロンプトは `{トリガー}, ...` 形式

**トラブルシュート**:
- `gated` エラー → HFでライセンス承認が済んでいるか確認
- T4でOOM → 無料枠では厳しいので L4/A100 を推奨
- 学習が途中で止まる → 再実行で続きから (チェックポイント自動再開)
